# Evaluation

- Here, we calculate the **CER (Character Error Rate)** of the Qwen2.5-VL generated annotation data that we use for fine-tuning by comparing with the box text annotations that comes with the SROIE v2 dataset.
- So, we are doing `CER(qwen_annotation_data,sroie_box_data)`

In [2]:
!pip install jiwer

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 31.5 MB/s eta 0:00:00


In [3]:
import glob
import jiwer

## 1. Load dataset

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [16]:
!unzip -q /content/drive/MyDrive/VLM/OCR-project.zip

#### 1.1. Load box data

In [65]:
sroie_box_train = glob.glob('./input/sroie_v2/SROIE2019/train/box/*.txt')
sroie_box_test = glob.glob('./input/sroie_v2/SROIE2019/test/box/*.txt')

sroie_box_train.sort()
sroie_box_test.sort()

print(len(sroie_box_train))
print(len(sroie_box_test))

# Example
print(sroie_box_train[1])
sample_file = open(sroie_box_train[1]).readlines()
print(sample_file)

626
347
./input/sroie_v2/SROIE2019/train/box/X00016469619.txt
['76,50,323,50,323,84,76,84,TAN WOON YANN\n', '110,165,315,165,315,188,110,188,INDAH GIFT & HOME DECO\n', '126,191,297,191,297,214,126,214,27,JALAN DEDAP 13,\n', '129,218,287,218,287,236,129,236,TAMAN JOHOR JAYA,\n', '100,243,324,243,324,261,100,261,81100 JOHOR BAHRU,JOHOR.\n', '70,268,201,268,201,285,70,285,TEL:07-3507405\n', '221,267,356,267,356,286,221,286,FAX:07-3558160\n', '177,317,241,317,241,336,177,336,RECEIPT\n', '16,364,257,364,257,392,16,392,19/10/2018 20:49:59 #01\n', '25,395,132,395,132,412,25,412,CASHIER: CN\n', '181,393,390,393,390,410,181,410,LOCATION/SP: 05 /0531\n', '26,445,129,445,129,463,26,463,MB: MO26588\n', '25,468,127,468,127,489,25,489,ROOM NO: 01\n', '286,466,400,466,400,488,286,488,050100035279\n', '24,518,114,518,114,538,24,538,DESC/ITEM\n', '182,519,213,519,213,538,182,538,QTY\n', '239,518,288,518,288,534,239,534,PRICE\n', '308,517,371,517,371,534,308,534,AMT(RM)\n', '25,543,273,543,273,559,25,55

The data is of the format:
```
['box coordinates,text\n','box coordinates,text\n',...]
```
- The box coordinates are the `(x,y)` coordinates of each of the 4 points of the bounding box
  - E.g. `76,50,323,50,323,84,76,84`
- text is the text inside that box
  - E.g. `TAN WOON YANN`

In [66]:
# Seperate the box coordinates from the text content
final_content = ''
for line in sample_file:
    final_content += line.split(',')[8] # get the text after 8th comma

print(final_content)

TAN WOON YANN
INDAH GIFT & HOME DECO
27TAMAN JOHOR JAYA81100 JOHOR BAHRUTEL:07-3507405
FAX:07-3558160
RECEIPT
19/10/2018 20:49:59 #01
CASHIER: CN
LOCATION/SP: 05 /0531
MB: MO26588
ROOM NO: 01
050100035279
DESC/ITEM
QTY
PRICE
AMT(RM)
ST-PRIVILEGE CARD/GD INDAH
88888
1
10.00
10.00
GF-TABLE LAMP/STITCH <I>
62483
1
55.90
55.90
@DISC
10.00%
-5.59
#TOTAL QTY
2
TOTAL AMT................. RM
60.31
ROUNDING ADJ............
-0.01
RM
60.30
CASH.................... RM
70.30
CHANGE.................. RM
10.00
THANK YOU ! PLEASE COME AGAIN !
GOODS SOLD ARE NOT RETURNABLE
THANK YOU ! FLEASE COME AOSIN !
GOODS SOLD ARE NOT RETURNABLE
DEALING IN WHOLESALE AND RETAIL.



#### 1.2. Load Annotation data (Ground truth)

In [67]:
qwen_annots_train = glob.glob('./input/qwen2_5_vl_3b_annots/train_annots/*.txt')
qwen_annots_test = glob.glob('./input/qwen2_5_vl_3b_annots/test_annots/*.txt')

qwen_annots_train.sort()
qwen_annots_test.sort()

print(len(qwen_annots_train))
print(len(qwen_annots_test))

626
347


## 2. Calculate CER

### 2.1. Calculate CER of Training Data

In [68]:
# Transformer to reduce multiple new lines to single new line.
tfms_multline = jiwer.Compose(
    [
        jiwer.SubstituteWords({"\n\n": "\n"})
    ]
)

def process_qwen_annots_data(file_paths, transformer):
    """
    Reads data from a list of file paths, converts to lowercase,
    applies a transformation, and returns the processed data.

    :param file_paths: List of file paths to process.
    :param transformer: A transformation function to apply to the data.
    :return: List of processed data strings.
    """
    processed_data = []
    for file_path in file_paths:
        with open(file_path) as f:
            data = f.read()
        processed_data.append(data.lower())
    return transformer(processed_data)

# usage
qwen_annots_data_train = process_qwen_annots_data(qwen_annots_train, tfms_multline)
print(qwen_annots_data_train[0])

tan woon yann
book ta .k (taman daya) sdn bhd
789417-w
no.5: 55,57 & 59, jalan sagu 1b,
taman daya,
81100 johor bahru,
johor.
document no : td01167104
date : 25/12/2018 8:13:39 pm
cashier : manis
member :
cash bill
code/desc price disc amount
qty rm
95569390-h0118 rf modelling clay kiddy fish
1 pc * 9.000 0.00 9.00
total : 9.00
rounding adjustment : 0.00
round d total (rm): 9.00
cash 90.00
change 0.00
goods sold are not returnable or
exchangeable
如有问题,敬请原谅。谢谢!
thank you
please come again!


In [69]:
def process_sroie_box_data(file_paths):
    """
    Reads SROIE box data from a list of file paths, extracts text content,
    converts to lowercase, and handles potential errors.

    :param file_paths: List of file paths to process.
    :return: Tuple containing a list of processed data strings and a list of indices
             that caused errors during processing.
    """
    processed_data = []
    error_ids = []
    for i, file_path in enumerate(file_paths):
        # Try-except because there are a few emtpy files and an
        # UTF-8 encoding error in one of the files.
        try:
            data = open(file_path).readlines()
            final_content = ''
            for line in data:
                final_content += line.rsplit(',', 1)[-1]

            processed_data.append(final_content.lower())
        except:
            error_ids.append(i)
            print(f"Erroneous index: {i}")
    return processed_data, error_ids

#usage
sroie_box_data_train, error_ids_train = process_sroie_box_data(sroie_box_train)
print(sroie_box_data_train[0])
print(error_ids_train) #will mostly be empty

tan woon yann
book ta .k(taman daya) sdn bnd
789417-w



johor.
document no : td01167104
date:
25/12/2018 8:13:39 pm
cashier:
manis
member:
cash bill
code/desc
price
disc
amount
qty
rm
rm
9556939040116
kf modelling clay kiddy fish
1 pc
*
9.000
0.00
9.00
total:
rour ding adjustment:
0.00
round d total (rm):
9.00
cash
10.00
change
1.00
goods sold are not returnable or
exchangeable
***
***
thank you
please come again !
9.00

[]


In [70]:
# Pop the data from the same indices in the qwen annots
for error_id in error_ids_train:
    qwen_annots_data_train.pop(error_id)
    print(f"Poped index: {error_id}")

In [71]:
## Calculate CER
error = jiwer.cer(sroie_box_data_train, qwen_annots_data_train)
print(f"CER: {error}")

CER: 0.30447570232717797


### 2.2. Calculate CER of Test Data

In [72]:
qwen_annots_data_test = process_qwen_annots_data(qwen_annots_test, tfms_multline)
print(qwen_annots_data_test[0])

tan chay yee
*** copy ***
ojc marketing sdn bhd
roc no: 538358-h
no 2 & 4, jalan bayu 4,
bandar seri alam,
81750 masai, johor
tel:07-388 2218 fax:07-388 8218
email: ng@ojcgroup.com
tax invoice
invoice no : pegiv-1030765
date : 15/01/2019 11:05:16 am
cashier : ng chuan min
sales persor : fatin
bill to : the peak quarry works
address :
description qty price amount
0000000111 1 193.00 193.00 sr
kings safety shoes kwd 805
qty: 1 total exclude gst: 193.00
total gst @6%: 0.00
total inclusive gst: 193.00
round amt: 0.00
total: 193.00
visa card 193.00
xxxxxx 4318
approval code:000
goods sold are not returnable & refundable
****thank you. please come again.****


In [73]:
sroie_box_data_test, error_ids_test = process_sroie_box_data(sroie_box_test)
print(sroie_box_data_test[0])
print(error_ids_test) #will mostly be empty

Erroneous index: 230
tan chay yee
*** copy ***
ojc marketing sdn bhd
roc no: 538358-h


 johor
tel:07-388 2218 fax:07-388 8218
email:ng@ojcgroup.com
tax invoice
invoice no
: pegiv-1030765
date
: 15/01/2019 11:05:16 am
cashier
: ng chuan min
sales person : fatin
bill to
: the peak quarry works
address
:.
description
qty
price
amount
000000111
1
193.00
193.00 sr
kings safety shoes kwd b05
qty: 1
total exclude gst:
193.00
total gst @6%:
0.00
total inclusive gst:
193.00
round amt:
0.00
total:
193.00
visa card
193.00
xxxxxxxxxxxx4318
approval code:000
goods sold are not returnable & refundable
****thank you. please come again.****

[230]


In [74]:
# Pop the data from the same indices in the qwen annots.
for error_id in error_ids_test:
    qwen_annots_data_test.pop(error_id)
    print(f"Poped index: {error_id}")

Poped index: 230


In [75]:
## Calculate CER
error = jiwer.cer(sroie_box_data_test, qwen_annots_data_test)
print(f"CER: {error}")

CER: 0.337988959741125
